In [1]:
#!pip3 install -U pymongo 
from pymongo import MongoClient
client = MongoClient('localhost',27017) 
db = client.apan5400
collection = db.collisions 

print("Clearing old collisions data")
collection.delete_many({})
print("Collision Collection cleared.\n")

Clearing old collisions data
Collision Collection cleared.



In [2]:
from pymongo import MongoClient
import requests

client = MongoClient("localhost", 27017)
db = client.apan5400
collisions_collection = db.collisions

print("Clearing old collisions data")
collisions_collection.delete_many({})
print("Collision collection cleared\n")

# NYC Open Data
url = "https://data.cityofnewyork.us/resource/h9gi-nx95.json"


limit = 50000  # Limited do to file size constraints
offset = 0
all_collisions = []

while True:
    params = {
        "$limit": limit,
        "$offset": offset,
        "$where": "crash_date >= '2021-01-01T00:00:00' AND crash_date <= '2024-12-31T23:59:59'"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        if not data:
            break


        all_collisions.extend(data)
        print(f"Fetched batch: {len(data)} records (Total fetched: {len(all_collisions)})")
        
        offset += limit
    else:
        print(f"Error: {response.status_code}")
        break

print(f"Total records fetched: {len(all_collisions)}")

# final load
if all_collisions:
    collisions_collection.insert_many(all_collisions)
    print("Total records inserted into MongoDB:", collisions_collection.count_documents({}))
else:
    print("No collisions data found")


Clearing old collisions data
Collision collection cleared

Fetched batch: 50000 records (Total fetched: 50000)
Fetched batch: 50000 records (Total fetched: 100000)
Fetched batch: 50000 records (Total fetched: 150000)
Fetched batch: 50000 records (Total fetched: 200000)
Fetched batch: 50000 records (Total fetched: 250000)
Fetched batch: 50000 records (Total fetched: 300000)
Fetched batch: 50000 records (Total fetched: 350000)
Fetched batch: 50000 records (Total fetched: 400000)
Fetched batch: 2365 records (Total fetched: 402365)
Total records fetched: 402365
Total records inserted into MongoDB: 402365


In [3]:
# head of first 3 rows
print("\n--- First 3 Records ---")
for doc in collection.find().limit(3):
    print(doc)
    print("-" * 50)


--- First 3 Records ---
{'_id': ObjectId('69322d775cca2387a9da6c9b'), 'crash_date': '2021-01-01T00:00:00.000', 'crash_time': '20:00', 'latitude': '40.83398', 'longitude': '-73.82635', 'location': {'latitude': '40.83398', 'longitude': '-73.82635', 'human_address': '{"address": "", "city": "", "state": "", "zip": ""}'}, 'on_street_name': 'BRUCKNER EXPRESSWAY', 'number_of_persons_injured': '0', 'number_of_persons_killed': '0', 'number_of_pedestrians_injured': '0', 'number_of_pedestrians_killed': '0', 'number_of_cyclist_injured': '0', 'number_of_cyclist_killed': '0', 'number_of_motorist_injured': '0', 'number_of_motorist_killed': '0', 'contributing_factor_vehicle_1': 'Pavement Slippery', 'collision_id': '4491746', 'vehicle_type_code1': 'Station Wagon/Sport Utility Vehicle'}
--------------------------------------------------
{'_id': ObjectId('69322d775cca2387a9da6c9c'), 'crash_date': '2021-01-01T00:00:00.000', 'crash_time': '5:28', 'latitude': '40.6873', 'longitude': '-73.973656', 'locatio

In [4]:
#fetching weather data
#free API only lets you download up to 1 year. this is the workaround.
import json
from datetime import datetime, timedelta
import time

collection_weather = db.weather

print("Clearing old weather data...")
collection_weather.delete_many({})
print("Weather Collection cleared.\n")

noaa_token = "FrpghyLrOmYismWZNYtqwlGqRPtyhNHm"
station_id = "GHCND:USW00094728"
url_weather = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
headers = {"token": noaa_token}

def generate_month_ranges(start_year, end_year):
    ranges = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            start_date = datetime(year, month, 1)
            
            if month == 12:
                end_date = datetime(year, 12, 31)
            else:
                end_date = datetime(year, month + 1, 1) - timedelta(days=1)
            
            ranges.append((start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")))
    
    return ranges

month_ranges = generate_month_ranges(2021, 2024)
all_weather_data = []

print(f"Fetching {len(month_ranges)} months of weather data...\n")

for start_date, end_date in month_ranges:
    print(f"Fetching {start_date} to {end_date}...")
    
    offset = 1
    max_retries = 3
    
    while True:
        params = {
            "datasetid": "GHCND",
            "stationid": station_id,
            "startdate": start_date,
            "enddate": end_date,
            "limit": 1000,
            "offset": offset,
            "units": "standard"
        }
        
        retry_count = 0
        success = False
        
        # Retry logic to avoid 503 errors, which can happen when downloading large amt of data 
        while retry_count < max_retries and not success:
            response = requests.get(url_weather, headers=headers, params=params)
            
            if response.status_code == 200:
                data = response.json()
                
                if "results" not in data or not data["results"]:
                    success = True
                    break
                
                all_weather_data.extend(data["results"])
                print(f"  Fetched {len(data['results'])} records (Total: {len(all_weather_data)})")
                success = True
                offset += 1000
                
            elif response.status_code == 429:
                print("  Rate limit hit, waiting 2 seconds...")
                time.sleep(2)
                retry_count += 1
                
            elif response.status_code == 503:
                retry_count += 1
                if retry_count < max_retries:
                    print(f"  503 Error - Retry {retry_count}/{max_retries} in 3 seconds...")
                    time.sleep(3)
                else:
                    print(f"  503 Error - Max retries reached, skipping this batch")
                    success = True
                    
            else:
                print(f"  Error: {response.status_code}")
                success = True
                break
        
        if not success or (response.status_code == 200 and ("results" not in data or not data["results"])):
            break
    
    time.sleep(0.5)  # The site can tell if you run too many queries, so we stagger our queries

# Reorganize by date
print("\nReorganizing weather data by date...")
weather_by_date = {}

for record in all_weather_data:
    date = record['date'].split('T')[0]
    datatype = record['datatype']
    value = record['value']
    
    if date not in weather_by_date:
        weather_by_date[date] = {
            'date': date,
            'station': record['station']
        }
    
    datatype_mapping = {
        'TMAX': 'max_temp_f',
        'TMIN': 'min_temp_f',
        'TAVG': 'avg_temp_f',
        'PRCP': 'precipitation_in',
        'SNOW': 'snowfall_in',
        'SNWD': 'snow_depth_in',
        'AWND': 'avg_wind_speed_mph',
        'WSF2': 'fastest_2min_wind_mph',
        'WSF5': 'fastest_5sec_wind_mph',
        'WESD': 'water_equivalent_snow_in'
    }
    
    if datatype in datatype_mapping:
        field_name = datatype_mapping[datatype]
        
        if 'temp' in field_name and datatype in ['TMAX', 'TMIN', 'TAVG']:
            value = (value / 10.0 * 9/5) + 32
        elif datatype == 'PRCP':
            value = value / 254.0
        elif datatype in ['SNOW', 'SNWD']:
            value = value / 25.4
        
        weather_by_date[date][field_name] = round(value, 2)

organized_weather = list(weather_by_date.values())

if organized_weather:
    collection_weather.insert_many(organized_weather)
    print(f"\n✓ Total weather days inserted: {len(organized_weather)}")
    
    print("\n--- First 3 Weather Records ---")
    for doc in collection_weather.find().limit(3):
        print(json.dumps(doc, indent=2, default=str))
        print("-" * 50)
else:
    print("\nNo weather data retrieved")

print(f"\nExpected: ~1,461 days (2021-2024)")
print(f"Actual: {collection_weather.count_documents({})} days")

Clearing old weather data...
Weather Collection cleared.

Fetching 48 months of weather data...

Fetching 2021-01-01 to 2021-01-31...
  Fetched 543 records (Total: 543)
Fetching 2021-02-01 to 2021-02-28...
  Fetched 509 records (Total: 1052)
Fetching 2021-03-01 to 2021-03-31...
  Fetched 543 records (Total: 1595)
Fetching 2021-04-01 to 2021-04-30...
  Fetched 499 records (Total: 2094)
Fetching 2021-05-01 to 2021-05-31...
  Fetched 544 records (Total: 2638)
Fetching 2021-06-01 to 2021-06-30...
  Fetched 526 records (Total: 3164)
Fetching 2021-07-01 to 2021-07-31...
  Fetched 571 records (Total: 3735)
Fetching 2021-08-01 to 2021-08-31...
  Fetched 541 records (Total: 4276)
Fetching 2021-09-01 to 2021-09-30...
  Fetched 469 records (Total: 4745)
Fetching 2021-10-01 to 2021-10-31...
  Fetched 524 records (Total: 5269)
Fetching 2021-11-01 to 2021-11-30...
  Fetched 516 records (Total: 5785)
Fetching 2021-12-01 to 2021-12-31...
  Fetched 549 records (Total: 6334)
Fetching 2022-01-01 to 2022-

In [5]:
#collection.drop()
#client.close()

In [6]:
from pymongo import MongoClient

client = MongoClient("localhost", 27017)
db = client.apan5400

print("Collections:", db.list_collection_names())
print("Collisions:", db.collisions.count_documents({}))
print("Weather:", db.weather.count_documents({}))


Collections: ['collisions', 'Companies', 'weather', 'cities', 'resumes']
Collisions: 402365
Weather: 1461


In [7]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient("localhost", 27017)
db = client.apan5400

collisions_df = pd.DataFrame(list(db.collisions.find({})))
weather_df = pd.DataFrame(list(db.weather.find({})))

print("Collisions rows:", len(collisions_df))
print("Weather rows:", len(weather_df))

collision_cols = [
    "crash_date",
    "crash_time",
    "borough",
    "zip_code",
    "latitude",
    "longitude",
    "on_street_name",
    "number_of_persons_injured",
    "number_of_persons_killed",
    "number_of_pedestrians_injured",
    "number_of_pedestrians_killed",
    "number_of_cyclist_injured",
    "number_of_cyclist_killed",
    "number_of_motorist_injured",
    "number_of_motorist_killed",
]


collision_cols = [c for c in collision_cols if c in collisions_df.columns]

collisions_clean = collisions_df[collision_cols].copy()

# Save to JSON files
collisions_clean.to_json(
    "collisions_raw.json",
    orient="records",
    default_handler=str
)
weather_df.to_json(
    "weather_raw.json",
    orient="records",
    default_handler=str
)

print("Files written: collisions_raw.json, weather_raw.json")


Collisions rows: 402365
Weather rows: 1461
Files written: collisions_raw.json, weather_raw.json


In [8]:
# Convert MongoDB ObjectId to string for collisions
if "_id" in collisions_clean.columns:
    collisions_clean["_id"] = collisions_clean["_id"].astype(str)

# Convert MongoDB ObjectId to string for weather
if "_id" in weather_df.columns:
    weather_df["_id"] = weather_df["_id"].astype(str)

In [9]:
#pip install sqlalchemy psycopg2-binary
#!pip install pyspark

In [10]:
from sqlalchemy import create_engine


POSTGRES_USER = "postgres"
POSTGRES_PASSWORD = "123"
POSTGRES_DB = "postgres"
POSTGRES_HOST = "localhost"
POSTGRES_PORT = "5432"

engine = create_engine(
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

# Collisions dataframe
collisions_clean.to_sql(
    "collisions",
    engine,
    if_exists="replace",
    index=False
)

# Weather dataframe
weather_df.to_sql(
    "weather",
    engine,
    if_exists="replace",
    index=False
)

print("Data uploaded to Postgres successfully!")

Data uploaded to Postgres successfully!


In [11]:
#!pip install libpq
#!pip install branca

In [12]:
from sqlalchemy import text
import pandas as pd


query = """
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'collisions';
"""

df = pd.read_sql(text(query), engine)
print(df)

                      column_name
0                      crash_date
1                      crash_time
2                         borough
3                        zip_code
4                        latitude
5                       longitude
6                  on_street_name
7       number_of_persons_injured
8        number_of_persons_killed
9   number_of_pedestrians_injured
10   number_of_pedestrians_killed
11      number_of_cyclist_injured
12       number_of_cyclist_killed
13     number_of_motorist_injured
14      number_of_motorist_killed


In [13]:
import json
import folium
from folium.features import GeoJsonTooltip
from folium.plugins import Fullscreen
from sqlalchemy import text